# Geordie Miner — runnable demo

Run the **full pipeline from a notebook**, with every config value inline at the top so you can tweak and re-run without touching `config/config.txt`.

**How to use:**

1. Set `DATA_DIR` and `RUN_NAME` in the config cell below.
2. Tweak any other parameters you want (or leave the defaults).
3. **Run All Cells** (or step through them one at a time).
4. Read the results at the bottom.

Each notebook run produces a fresh `output/<RUN_NAME>/` folder, so you can run this notebook many times with different params and compare results.

## 1. Configuration (edit these values)

Everything the pipeline reads — paths, knobs, topic counts — lives in this one cell. Change a number, re-run from this cell down, see what happens.

In [ ]:
# === DATA + RUN NAME (the two you'll change most often) ===
DATA_DIR = "data/myproject"        # path to your input folder, relative to repo root
RUN_NAME = "demo_run"               # output goes to output/<RUN_NAME>/
STAGES   = "ingest,preprocess,terms,phrases,topics"   # comma-separated subset, or all

# === LANGUAGE + PREPROCESSING ===
LANGUAGE       = "english"          # NLTK stopword language
MIN_FREQUENCY  = 25                  # drop tokens that appear < N times in the corpus

# === TERM ANALYSIS ===
TOP_N_TERMS         = 200
OUTPUT_WORDCLOUD    = True
WORDCLOUD_WIDTH     = 800
WORDCLOUD_HEIGHT    = 400
WORDCLOUD_BG_COLOR  = "white"

# === PHRASE ANALYSIS ===
BIGRAM_THRESHOLD       = 10           # min frequency to be exported
TRIGRAM_THRESHOLD      = 10
BIGRAM_EXPORT_COUNT    = 100
TRIGRAM_EXPORT_COUNT   = 100
CLUSTERING_METRIC      = "jaccard"
LINKAGE_METHOD         = "ward"
DENDROGRAM_FIGSIZE     = "10, 7"
COOCCURRENCE_THRESHOLD = 15           # min edge weight in the network
WINDOW_SIZE            = 5            # sentence-local co-occurrence window

# === TOPIC MODELS ===
# Each model runs at K, K*MULT_1, K*MULT_2, K*MULT_3. Set a multiplier to 0 to skip that level.
TOPIC_MULT_1 = 2
TOPIC_MULT_2 = 3
TOPIC_MULT_3 = 4

KMEANS_TOPICS = 5

LDA_TOPICS          = 5
LDA_TERMS_PER_TOPIC = 10
LDA_PASSES          = 50
LDA_PER_WORD_TOPICS = 0

NMF_TOPICS          = 5
NMF_TERMS_PER_TOPIC = 10
NMF_MAX_ITER        = 1000

HDP_TERMS_PER_TOPIC = 10

print(f"Will process: {DATA_DIR}")
print(f"Output will go to: output/{RUN_NAME}")
print(f"Stages: {STAGES}")

## 2. Environment setup

Point Python at the project root and make `src/` importable. Safe to re-run — idempotent.

In [ ]:
import os, sys
from pathlib import Path

# Find the project root (the folder containing geordie_miner.py)
_here = Path.cwd()
PROJECT_ROOT = _here if (_here / "geordie_miner.py").exists() else _here.parent
os.chdir(PROJECT_ROOT)

src_path = str((PROJECT_ROOT / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Working directory: {os.getcwd()}")
print(f"src/ on path:      {src_path in sys.path}")

## 3. Materialise the config

Geordie Miner reads config from an `.ini`-style file. We write the values from cell 1 into a temporary config file and load it the same way the CLI does — so this notebook produces identical results to running `python geordie_miner.py run ...` from a terminal.

In [ ]:
TEMP_CONFIG = Path("config/_demo_config.txt")
TEMP_CONFIG.write_text(f"""\
[default]
language = {LANGUAGE}

[preprocessing]
stopwords_file = stopwords.txt
substitutions_file = substitutions.txt
min_frequency = {MIN_FREQUENCY}

[term_analysis]
top_n_terms = {TOP_N_TERMS}
output_wordcloud = {str(OUTPUT_WORDCLOUD).lower()}
wordcloud_width = {WORDCLOUD_WIDTH}
wordcloud_height = {WORDCLOUD_HEIGHT}
wordcloud_background_color = {WORDCLOUD_BG_COLOR}

[phrase_analysis]
bigram_threshold = {BIGRAM_THRESHOLD}
trigram_threshold = {TRIGRAM_THRESHOLD}
bigram_export_count = {BIGRAM_EXPORT_COUNT}
trigram_export_count = {TRIGRAM_EXPORT_COUNT}
clustering_metric = {CLUSTERING_METRIC}
linkage_method = {LINKAGE_METHOD}
dendrogram_figsize = {DENDROGRAM_FIGSIZE}
cooccurrence_threshold = {COOCCURRENCE_THRESHOLD}
window_size = {WINDOW_SIZE}

[topic_modelling]
topic_modelling_multi1 = {TOPIC_MULT_1}
topic_modelling_multi2 = {TOPIC_MULT_2}
topic_modelling_multi3 = {TOPIC_MULT_3}

[topic_kmeans]
kmeans_topics = {KMEANS_TOPICS}

[topic_lda]
lda_topics = {LDA_TOPICS}
lda_terms_per_topic = {LDA_TERMS_PER_TOPIC}
lda_passes = {LDA_PASSES}
lda_per_word_topics = {LDA_PER_WORD_TOPICS}

[topic_nmf]
nmf_topics = {NMF_TOPICS}
nmf_terms_per_topic = {NMF_TERMS_PER_TOPIC}
nmf_max_iter = {NMF_MAX_ITER}

[topic_hdp]
terms_per_topic_hdp = {HDP_TERMS_PER_TOPIC}
""")
print(f"Temporary config written: {TEMP_CONFIG}")

## 4. Run the pipeline

This is the long cell. Stages stream their progress to the notebook output as they run. Expect ~30 seconds for a tiny corpus, ~25 minutes for ~100 papers (most of it is the coherence stage at the end).

If you only want a quick smoke test, change `STAGES` in cell 1 to `"ingest,preprocess,terms"` and re-run.

In [ ]:
from config import load_config
import geordie_miner

cfg = load_config(str(TEMP_CONFIG), DATA_DIR, run_name=RUN_NAME)
stages = [s.strip() for s in STAGES.split(",") if s.strip()]
geordie_miner.run_pipeline(cfg, stages)

## 5. Results

Everything below loads from `output/<RUN_NAME>/`. Re-run individual cells to refresh specific artefacts.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = Path("output") / RUN_NAME
print(f"Loading results from: {OUT}")

### Corpus statistics

In [ ]:
stats = OUT / "corpus_stats.txt"
print(stats.read_text() if stats.exists() else "(no corpus_stats.txt — was 'ingest' / 'preprocess' in STAGES?)")

### Top lemmatised terms

In [ ]:
terms_csv = OUT / "terms_lemmatised.csv"
if terms_csv.exists():
    display(pd.read_csv(terms_csv).head(30))
else:
    print("(no terms_lemmatised.csv — was 'terms' in STAGES?)")

### Word clouds

In [ ]:
for name in ("wordcloud_lemmatised.jpg", "wordcloud_raw.jpg"):
    path = OUT / name
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

### Top phrases

In [ ]:
for fname in ("bigrams.csv", "trigrams.csv"):
    path = OUT / fname
    if path.exists():
        display(Markdown(f"**{fname}**"))
        display(pd.read_csv(path).head(15))

### Topic models

In [ ]:
for path in sorted(OUT.glob("topics_*.txt")):
    display(Markdown(f"**{path.name}**"))
    print(path.read_text())

### Coherence scores (higher c_v = better)

In [ ]:
coh = OUT / "coherence_scores.csv"
if coh.exists():
    df = pd.read_csv(coh).sort_values("coherence_c_v", ascending=False)
    display(df)
    best = df.iloc[0]
    display(Markdown(f"**Best model on this corpus:** `{best['model']}` at K={int(best['K'])} (c_v = {best['coherence_c_v']:.3f})"))
else:
    print("(no coherence_scores.csv — was 'topics' in STAGES?)")

### Top documents per topic

For each topic in each model, the 5 most representative papers. Filter the table below to read the actual papers behind any theme you find interesting.

In [ ]:
tdocs = OUT / "topic_top_docs.csv"
if tdocs.exists():
    df = pd.read_csv(tdocs)
    # Show one example slice: the first model/K combination
    if not df.empty:
        first = df.iloc[0]
        subset = df[(df["model"] == first["model"]) & (df["K"] == first["K"])]
        display(Markdown(f"**Example: top documents for {first['model']} (K={int(first['K'])})**"))
        display(subset)
        display(Markdown("Adjust `df[(df['model'] == ...) & (df['K'] == ...)]` to view other models."))
else:
    print("(no topic_top_docs.csv — was 'topics' in STAGES?)")

### Dendrogram + network

In [ ]:
dendro = OUT / "dendrogram.png"
if dendro.exists():
    display(Image(filename=str(dendro)))

gexf = OUT / "network.gexf"
if gexf.exists():
    import networkx as nx
    g = nx.read_gexf(str(gexf))
    print(f"Co-occurrence network: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
    print(f"Open in Gephi for visual exploration: {gexf.resolve()}")

## Next steps

- **Pick the best K.** Look at the coherence table above — the model/K with the highest `c_v` is the best topic model on this corpus. Read those topic words and the top documents per topic.
- **Tweak and re-run.** Change values in cell 1, then *Cell → Run All Below* from cell 3. Each run replaces `output/<RUN_NAME>/`.
- **Compare configurations.** Change `RUN_NAME` to something new (e.g. `"k20_aggressive"`) and run again. Both runs are preserved side-by-side. Then from a terminal:
  ```bash
  python geordie_miner.py compare output/demo_run output/k20_aggressive
  ```
  That writes `output/comparison_report.md` showing the two side-by-side.